In this notebook, I'll re-implement a transformer-based machine translation algorithm. This will be based on Chapter 11 of the D2L.AI textbook. In doing so, I'll try to do everything from memory. This means not consulting the D2L.AI textbook or the code I wrote previously.

# 1. Get Dataset

We'll use the FraEngMT dataset. We've already written code to extract batches from this dataset in `utils.data` and won't rewrite the cleaning, tokenization, and extraction algorithms from scratch.

In [1]:
import os
import sys
print(os.getcwd())
sys.path.append(os.path.join(os.getcwd(), '..'))
from utils.data import FraEngMTDataset, get_data_mt

/Users/nickmcgreivy/code/ml-practice/ml-experiments/natural_language_processing


In [2]:
batch_size = 8
num_steps = 10
num_train = 1024
num_val = 512
train_dl, val_dl, src_vocab, tgt_vocab = get_data_mt(batch_size, num_steps, num_train, num_val)

The dataloaders return 4 arrays: `src`, `tgt`, `src_valid_len`, `tgt_label`

In [4]:
for src, tgt, src_valid_len, tgt_label in train_dl:
    print(src.shape, tgt.shape, src_valid_len.shape, tgt_label.shape)
    print(src[0], tgt[0], src_valid_len[0], tgt_label[0])
    print(src_vocab.to_tokens(src[0]))
    print(tgt_vocab.to_tokens(tgt[0]))
    print(tgt_vocab.to_tokens(tgt_label[0]))
    break

torch.Size([8, 10]) torch.Size([8, 10]) torch.Size([8]) torch.Size([8, 10])
tensor([341, 322,   2,   4,   5,   5,   5,   5,   5,   5]) tensor([ 3,  6, 68,  2,  4,  5,  5,  5,  5,  5]) tensor(4) tensor([ 6, 68,  2,  4,  5,  5,  5,  5,  5,  5])
['use', 'this', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
['<bos>', '<unk>', 'ceci', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']
['<unk>', 'ceci', '.', '<eos>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>', '<pad>']


# 2. Create Encoder-Decoder Abstract Class

Our model will use an encoder-decoder architecture. The encoder will return a tuple of outputs, and then those outputs will be fed as inputs to the decoder. 

These will subclass `models.Module`, which allows for my pre-implemented plotting routines during training.

In [5]:
from abc import ABC
from utils.models import Module

In [6]:
class Encoder(Module, ABC):
    def __init__(self):
        super().__init__()

    def forward(self):
        pass

In [7]:
class Decoder(Module, ABC):
    def __init__(self):
        super().__init__()

    def init_state(self):
        pass

    def forward(self):
        pass

In [8]:
class EncoderDecoder(Module):
    def __init__(self, encoder, decoder):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
    
    def forward(self, src, tgt, *args):
        enc_state = self.encoder(src, *args)
        dec_init_state = self.decoder.init_state(enc_state, *args)
        return self.decoder(tgt, dec_init_state, *args)

# 3. Implement Transformer Encoder Block

In [9]:
import math

import torch
from torch import nn
import torch.nn.functional as F

In [10]:
def softmax(X):
    exp = torch.exp(X)
    return exp / exp.sum(dim=-1, keepdim=True)

def masked_softmax(X, valid_lens=None, neg_inf=-1e6):
    """Perform masked softmax of X, with mask given by valid_lens

    Mask is along last dimension (M) or last two dimensions (N, M)
    
    Args
        X (torch.Tensor): input of floats of shape (B, N, M) 
        valid_lens (torch.Tensor): mask of ints of shape (B,) or (B, N)
    
    Returns
        (torch.Tensor): 
    """
    if valid_lens is None:
        return softmax(X)
    else:
        B, N, M = X.shape
        if valid_lens.dim() == 1:
            valid_lens = torch.repeat_interleave(valid_lens[:, None], N, dim=1)

        X = X.reshape(-1, M)
        valid_lens = valid_lens.reshape(-1)

        mask = (torch.arange(0, M)[None, :] >= valid_lens[:, None])
        X[mask] = neg_inf
        X = X.reshape(B, N, M)

        return softmax(X)

In [11]:
# test masked_softmax

X = torch.randn(3, 5, 8)
valid_lens = torch.tensor([3,6,7])
print(masked_softmax(X, valid_lens).shape)
print(masked_softmax(X, valid_lens))

torch.Size([3, 5, 8])
tensor([[[0.2048, 0.5449, 0.2502, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.8553, 0.0322, 0.1125, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4033, 0.3169, 0.2797, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0449, 0.0087, 0.9464, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.2511, 0.1801, 0.5689, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.1494, 0.3018, 0.1714, 0.0691, 0.0285, 0.2798, 0.0000, 0.0000],
         [0.1928, 0.3343, 0.1610, 0.0245, 0.0394, 0.2480, 0.0000, 0.0000],
         [0.1925, 0.0305, 0.0286, 0.0474, 0.1484, 0.5527, 0.0000, 0.0000],
         [0.3235, 0.0436, 0.2185, 0.1249, 0.1359, 0.1535, 0.0000, 0.0000],
         [0.0715, 0.2008, 0.0287, 0.0699, 0.5880, 0.0409, 0.0000, 0.0000]],

        [[0.0060, 0.0433, 0.1162, 0.1983, 0.5708, 0.0079, 0.0575, 0.0000],
         [0.0434, 0.2569, 0.0134, 0.0438, 0.0751, 0.3864, 0.1810, 0.0000],
         [0.1651, 0.3016, 0.1664, 0.1717, 0.1180, 0.0297, 0.0475, 0.0000],

In [12]:
# test masked_softmax

X = torch.randn(3, 5, 8)
valid_lens = torch.tensor([[1,2,3,4,5],[3,3,4,1,2],[4,4,4,3,4]])
print(masked_softmax(X, valid_lens).shape)
print(masked_softmax(X, valid_lens))

torch.Size([3, 5, 8])
tensor([[[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5865, 0.4135, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.5773, 0.3498, 0.0729, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.4315, 0.1581, 0.2972, 0.1133, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0495, 0.3433, 0.0558, 0.4109, 0.1404, 0.0000, 0.0000, 0.0000]],

        [[0.5971, 0.0662, 0.3367, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.1687, 0.3448, 0.4865, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.3773, 0.0431, 0.1159, 0.4636, 0.0000, 0.0000, 0.0000, 0.0000],
         [1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.7311, 0.2689, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.1466, 0.2835, 0.2419, 0.3280, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0554, 0.0822, 0.3163, 0.5461, 0.0000, 0.0000, 0.0000, 0.0000],
         [0.0616, 0.1747, 0.1286, 0.6350, 0.0000, 0.0000, 0.0000, 0.0000],

In [13]:
class DotProductAttention(Module):
    def __init__(self):
        super().__init__()
    
    def forward(self, Q, K, V, valid_lens):
        D = Q.shape[-1]
        dot_product = torch.bmm(Q, torch.transpose(K, 1, 2)) / math.sqrt(D)
        attention_matrix = masked_softmax(dot_product, valid_lens)
        return torch.bmm(attention_matrix, V)

In [14]:
# test DotProductAttention

X = torch.randn(4, 8, 32)
valid_lens = torch.tensor([3,5,6,7])
attention = DotProductAttention()
print(attention(X, X, X, valid_lens).shape)

torch.Size([4, 8, 32])


In [15]:
class MultiHeadAttention(Module):
    def __init__(self, num_hiddens, num_heads, dropout=0.5):
        super().__init__()
        self.num_heads = num_heads
        self.Wq = nn.LazyLinear(num_hiddens)
        self.Wk = nn.LazyLinear(num_hiddens)
        self.Wv = nn.LazyLinear(num_hiddens)
        self.Wo = nn.Linear(num_hiddens, num_hiddens)
        self.attention = DotProductAttention()

    def batch_heads(self, X):
        """(B, N, M) -> (B*h, N, M/h)"""
        assert len(X.shape) == 3
        X = X.reshape(X.shape[0], X.shape[1], -1, self.num_heads)
        X = torch.permute(X, (0, 3, 1, 2))
        return X.reshape(-1, X.shape[2], X.shape[3])
    
    def unbatch_heads(self, X):
        """(B*h, N, M/h) -> (B, N, M)"""
        assert len(X.shape) == 3
        X = X.reshape(-1, self.num_heads, X.shape[1], X.shape[2])
        X = torch.permute(X, (0, 2, 3, 1))
        return X.reshape(X.shape[0], X.shape[1], -1)

    def forward(self, Q, K, V, valid_lens):
        Q = self.batch_heads(self.Wq(Q))
        K = self.batch_heads(self.Wk(K))
        V = self.batch_heads(self.Wv(V))

        if valid_lens is not None:
            valid_lens = torch.repeat_interleave(
                valid_lens, self.num_heads, dim=0
            )

        attention_heads_output = self.attention(Q, K, V, valid_lens)
        return self.Wo(self.unbatch_heads(attention_heads_output))


In [16]:
# test MultiHeadAttention

attention = MultiHeadAttention(32, 4)
attention.eval()
X = torch.randn(4, 8, 32)
valid_lens = torch.tensor([3,5,6,7])
print(attention(X, X, X, valid_lens).shape)

torch.Size([4, 8, 32])


In [17]:
class PositionWiseFFN(Module):
    def __init__(self, num_hiddens, ffn_num_hiddens, dropout=0.5):
        super().__init__()
        self.linear1 = nn.Linear(num_hiddens, ffn_num_hiddens)
        self.linear2 = nn.Linear(ffn_num_hiddens, num_hiddens)
        
    def forward(self, X):
        X = F.relu(self.linear1(X))
        return self.linear2(X)

In [18]:
class TransformerEncoderBlock(Module):
    def __init__(self, num_hiddens, num_heads, ffn_num_hiddens, dropout=0.5):
        super().__init__()
        self.ln1 = nn.LayerNorm(num_hiddens)
        self.attention = MultiHeadAttention(num_hiddens, num_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(num_hiddens)
        self.ffn = PositionWiseFFN(num_hiddens, ffn_num_hiddens, dropout=dropout)

    def forward(self, X, valid_len):
        Xn = self.ln1(X)
        Y = X + self.attention(Xn, Xn, Xn, valid_len)
        return Y + self.ffn(self.ln2(Y))

In [19]:
# test MultiHeadAttention

attentionblk = TransformerEncoderBlock(32, 4, 16)
attentionblk.eval()
X = torch.randn(4, 8, 32)
valid_lens = torch.tensor([3,5,6,7])
print(attentionblk(X, valid_lens).shape)

torch.Size([4, 8, 32])


# 4. Implement Positional Encoding

In [24]:
class PositionalEncoding(Module):
    def __init__(self, num_hiddens, dropout=0.5, max_len=100):
        super().__init__()
        self.P = torch.zeros(max_len, num_hiddens)
        X = torch.arange(max_len)[:,None] / torch.pow(10000,
            torch.arange(0, num_hiddens, 2) / num_hiddens
        )[None,:]
        self.P[:, 0::2] = torch.sin(X)
        self.P[:, 1::2] = torch.cos(X)
        self.dropout = nn.Dropout(p=dropout)
    
    def forward(self, X):
        X = X + self.P[None, 0:X.shape[1], :].to(X.device)
        return self.dropout(X)

In [25]:
# test PositionalEncoding

posenc = PositionalEncoding(32)
X = torch.randn(4, 8, 32)
print(posenc(X).shape)

torch.Size([4, 8, 32])


# 5. Write Transformer-Based Encoder

We haven't implemented the details of the transformer block yet. We will implement those details after creating the structure of the model.

In [ ]:
class TransformerEncoder(Encoder):
    def __init__(self, num_hiddens, num_blks, num_heads, ffn_num_hiddens, src_vocab, dropout=0.5):
        super().__init__()
        self.num_hiddens = num_hiddens
        self.embedding = nn.Embedding(len(src_vocab), num_hiddens)
        self.positional_encoding = PositionalEncoding(num_hiddens, dropout=dropout)
        self.blks = nn.ModuleList()
        for i in range(num_blks):
            self.blks.append(TransformerEncoderBlock(num_hiddens, num_heads, ffn_num_hiddens, dropout=dropout))
        
    def forward(self, src, src_valid_len):
        X = self.embedding(src)
        X = self.positional_encoding(X * math.sqrt(self.num_hiddens))
        for blk in self.blks:
            X = blk(X, src_valid_len)
        return X

In [27]:
encoder = TransformerEncoder(32, 5, 4, 16, src_vocab, dropout=0.4)

for src, tgt, src_valid_len, tgt_label in train_dl:
    print(encoder(src, src_valid_len).shape)
    break

torch.Size([8, 10, 32])


# 6. Write Transfor-Based Decoder Block

In [ ]:
class TransformerDecoderBlock(Module):
    def __init__(self, num_hiddens, num_heads, ffn_num_hiddens, dropout=0.5):
        super().__init__()
        self.ln1 = nn.LayerNorm(num_hiddens)
        self.attention1 = MultiHeadAttention(num_hiddens, num_heads, dropout=dropout)
        self.ln2 = nn.LayerNorm(num_hiddens)
        self.attention2 = MultiHeadAttention(num_hiddens, num_heads, dropout=dropout)
        self.ln3 = nn.LayerNorm(num_hiddens)
        self.ffn = PositionWiseFFN(num_hiddens, ffn_num_hiddens, dropout=dropout)

    def forward(self, X, dec_init_state, src_valid_len, tgt_valid_len):
        Xn = self.ln1(X)
        Y1 = X + self.attention1(Xn, Xn, Xn, tgt_valid_len)
        Y1n = self.ln2(Y1)
        Y2 = Y1 + self.attention2(Y1n, dec_init_state, dec_init_state, src_valid_len)
        return Y2 + self.ffn(self.ln3(Y2))

# 6. Write Transformer-Based Decoder

The decoder takes the output of the encoder, passes it to `init_state`, then passes `tgt`, `dec_init_state`, and `src_valid_len` to the decoder. `tgt` is passed through an embedding layer, and both are passed to a sequence of `TransformerDecoderBlock`.

In [ ]:
class TransformerDecoder(Module):
    def __init__(self, num_hiddens, num_blks, num_heads, ffn_num_hiddens, tgt_vocab, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(len(tgt_vocab), num_hiddens)
        self.blks = nn.ModuleList()
        for i in range(num_blks):
            self.blks.append(TransformerDecoderBlock(num_hiddens, num_heads, ffn_num_hiddens, dropout=dropout))
        self.linear_out = nn.Linear(num_hiddens, len(tgt_vocab))
    
    def init_state(self, enc_output, src_valid_len):
        return enc_output
    
    def forward(self, tgt, dec_init_state, src_valid_len):
        X = self.embedding(tgt)
        tgt_valid_len = ...
        for blk in self.blks:
            X = blk(X, dec_init_state, src_valid_len, tgt_valid_len)
        return self.linear_out(X)

# 7. Train model

In [ ]:
from utils.train import fit_mt

In [ ]:
lr = 1e-3
num_hiddens = 64
ffn_num_hiddens = 32
num_blks = 4
num_heads = 8
num_epochs = 2

In [ ]:
encoder = TransformerEncoder(num_hiddens, num_blks, num_heads, ffn_num_hiddens, src_vocab)
decoder = TransformerDecoder(num_hiddens, num_blks, num_heads, ffn_num_hiddens, tgt_vocab)
model = EncoderDecoder(encoder, decoder)
opt = torch.optim.Adam(model.parameters(), lr=lr)

In [ ]:
def masked_loss_fn_mt(tgt, tgt_label, tgt_vocab):
    loss = F.cross_entropy(tgt, tgt_label, reduce='none')
    mask = (tgt_label != tgt_vocab['<pad>']).type(torch.float)
    return torch.mean(loss * mask)

loss_fn_mt = lambda tgt, tgt_label: masked_loss_fn_mt(tgt, tgt_label, tgt_vocab)

In [ ]:
fit_mt(train_dl, val_dl, model, opt, num_epochs=num_epochs, device='cpu', 
       id='TestScratchTransformerMachineTranslation', 
       loss_fn=loss_fn_mt)